In [ ]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt

# helper
def strip_utm(url):
    return re.sub(r"[?&]utm_source=chatgpt\.com", "", url)

# load
with open('aggregated_data.json','r',encoding='utf-8') as f:
    data = json.load(f)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Data preparation (as before)
har_counts = {cat: len(data[cat]) for cat in data}
series = pd.Series(har_counts).sort_index()

# Apply a softer muted style
#plt.style.use('seaborn-muted')          # harmonious base :contentReference[oaicite:19]{index=19}

# Plot setup
fig, ax = plt.subplots(figsize=(10, 6))

# Soft pastel colors from a qualitative palette
colors = plt.get_cmap('Pastel1')(np.arange(len(series)))  # pastel palette :contentReference[oaicite:20]{index=20}

# Create bars with defined width for clear gaps
bars = ax.bar(
    series.index,
    series.values,
    width=0.6,                         # narrower bars => ~50% spacing :contentReference[oaicite:21]{index=21}
    edgecolor='gray',
    linewidth=1.2,
    color=colors,
    zorder=3
)

# Zero baseline
ax.set_ylim(bottom=0)                  # start y-axis at zero :contentReference[oaicite:22]{index=22}

# Titles & labels
ax.set_title('HARs per Category', fontsize=16, pad=15)
ax.set_ylabel('Number of HARs', fontsize=14, labelpad=10)
ax.tick_params(axis='x', rotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Light dashed gridlines
ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)  # subtle grid :contentReference[oaicite:23]{index=23}

# Annotate bar values
for bar in bars:
    h = bar.get_height()
    ax.annotate(
        f'{h}',
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=11
    )  # direct labels :contentReference[oaicite:24]{index=24}

# plt.tight_layout()
# plt.show()
# create a color map for each category
color_map = dict(zip(series.index, colors))

# plot original bars
bars = ax.bar(
    series.index,
    series.values,
    width=0.6,
    edgecolor='gray',
    linewidth=1.2,
    color=[color_map[cat] for cat in series.index],
    zorder=3
)

# now add a stacked bar for the sum of instramental, navigational, and transactional
stack_cats = ['instramental', 'navigational', 'transactional']
total_stack = series[stack_cats].sum()
bottom = 0
for cat in stack_cats:
    val = series[cat]
    ax.bar(
        'Combined',
        val,
        bottom=bottom,
        width=0.6,
        edgecolor='gray',
        linewidth=1.2,
        color=color_map[cat],
        zorder=3
    )
    bottom += val

# annotate each segment in the combined bar
y = 0
for cat in stack_cats:
    val = series[cat]
    ax.annotate(
        f'{val}',
        xy=('Combined', y + val/2),
        xytext=(0, 0),
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=11,
        color='black'
    )
    y += val

# annotate the combined total above the bar
ax.annotate(
    f'{total_stack}',
    xy=('Combined', total_stack),
    xytext=(0, 5),
    textcoords='offset points',
    ha='center',
    va='bottom',
    fontsize=11,
    weight='bold'
)

# refresh layout and show
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Filter out HARs that have only Bing searches (no Google results)
for cat in list(data.keys()):
    data[cat] = {hid: rec for hid, rec in data[cat].items() if rec['google_urls']}

for cat in list(data.keys()):
    data[cat] = {hid: rec for hid, rec in data[cat].items() if rec['bing_urls']}

# Additionally filter out HAR IDs 87–110 for navigational and 160–175 for factual
# Assuming HAR keys like 'network-logs-prompt-<ID>_<timestamp>'
# if 'navigational' in data:
#     filtered = {}
#     for hid, rec in data['navigational'].items():
#         try:
#             num = int(hid.split('-')[3].split('_')[0])
#         except Exception:
#             filtered[hid] = rec
#             continue
#         if not (87 <= num <= 110):
#             filtered[hid] = rec
#     data['navigational'] = filtered
# if 'factual' in data:
#     filtered = {}
#     for hid, rec in data['factual'].items():
#         try:
#             num = int(hid.split('-')[3].split('_')[0])
#         except Exception:
#             filtered[hid] = rec
#             continue
#         if not (160 <= num <= 175):
#             filtered[hid] = rec
#     data['factual'] = filtered

# 1) Compute hit‐rate records and average per category with substring matching
records = []
for cat, har_dict in data.items():
    for har_id, rec in har_dict.items():
        prompts = rec['urls_from_prompt']
        search_urls = [e['url'] for e in rec['bing_urls'] + rec['google_urls']]
        total = len(prompts)
        hits = 0
        for p in prompts:
            # count as hit if any search URL equals p or is a substring of p
            if any(s == p or s in p for s in search_urls):
                hits += 1
        records.append({
            'category': cat,
            'har_id': har_id,
            'hit_rate': hits / total if total else 0
        })

df_hr = pd.DataFrame(records)
avg_hr = df_hr.groupby('category')['hit_rate'].mean().sort_index()


# 2. Apply muted style and pastel palette
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.get_cmap('Pastel1')(np.arange(len(avg_hr)))

bars = ax.bar(
    avg_hr.index,
    avg_hr.values,
    width=0.6,
    edgecolor='gray',
    linewidth=1.2,
    color=colors,
    zorder=3
)

# 3. Zero-based baseline
ax.set_ylim(bottom=0)

# 4. Titles & labels
ax.set_title('Average Hit Rate (Combine Search Engines) per Category', fontsize=16, pad=15)
ax.set_ylabel('Average Hit Rate', fontsize=14, labelpad=10)
ax.tick_params(axis='x', rotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# 5. Subtle horizontal gridlines
ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)

# 6. Annotate each bar with its value (formatted as percentage)
for bar in bars:
    h = bar.get_height()
    ax.annotate(
        f'{h:.1%}',
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=11
    )

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Prepare the data (as you already have)
b_records, g_records = [], []
for cat, har_dict in data.items():
    for har_id, rec in har_dict.items():
        prompts = set(rec['urls_from_prompt'])
        total = len(prompts)
        bing_urls = set(e['url'] for e in rec['bing_urls'])
        google_urls = set(e['url'] for e in rec['google_urls'])
        b_rate = len(prompts & bing_urls) / total if total else 0
        g_rate = len(prompts & google_urls) / total if total else 0
        b_records.append({'category': cat, 'rate_bing': b_rate})
        g_records.append({'category': cat, 'rate_google': g_rate})
df_b = pd.DataFrame(b_records)
df_g = pd.DataFrame(g_records)
avg_b = df_b.groupby('category')['rate_bing'].mean().sort_index()
avg_g = df_g.groupby('category')['rate_google'].mean().sort_index()

# 2. Styling setup
fig, ax = plt.subplots(figsize=(12, 6))

n = len(avg_b)
index = np.arange(n)
bar_width = 0.4
opacity = 0.9

# 3. Choose two distinct pastel colors
cmap = plt.get_cmap('Pastel1')
colors = cmap([0, 1])  # first two pastel shades

# 4. Plot grouped bars
bars_b = ax.bar(
    index - bar_width/2,
    avg_b.values,
    bar_width,
    label='Bing',
    edgecolor='gray',
    linewidth=1.2,
    color=colors[0],
    zorder=3
)
bars_g = ax.bar(
    index + bar_width/2,
    avg_g.values,
    bar_width,
    label='Google',
    edgecolor='gray',
    linewidth=1.2,
    color=colors[1],
    zorder=3
)

# 5. Zero baseline & grid
ax.set_ylim(0, 1)
ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)

# 6. Labels, title, legend
ax.set_title('Average Hit Rate per Category (Bing vs Google)', fontsize=16, pad=15)
ax.set_ylabel('Average Hit Rate', fontsize=14, labelpad=10)
ax.set_xticks(index)
ax.set_xticklabels(avg_b.index, rotation=45, ha='right', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=12)

# 7. Annotate bars
for bars in (bars_b, bars_g):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f'{h:.1%}',
            xy=(bar.get_x() + bar.get_width()/2, h),
            xytext=(0, 5),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=11
        )

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Compute overall hit rates (as you already have)
totals = {'bing': 0, 'google': 0, 'both': 0, 'prompts': 0}
for recs in data.values():
    for rec in recs.values():
        p = set(rec['urls_from_prompt'])
        totals['prompts'] += len(p)
        b = set(e['url'] for e in rec['bing_urls'])
        g = set(e['url'] for e in rec['google_urls'])
        totals['bing']   += len(p & b)
        totals['google'] += len(p & g)
        totals['both']   += len(p & (b | g))
rates = {k: totals[k] / totals['prompts'] for k in ('bing', 'google', 'both')}

# 2. Styling setup
fig, ax = plt.subplots(figsize=(8, 6))

series = pd.Series(rates).reindex(['bing','google','both'])
n = len(series)
index = np.arange(n)
bar_width = 0.6

# 3. Pastel1 palette for three bars
colors = plt.get_cmap('Pastel1')(np.arange(n))

bars = ax.bar(
    index,
    series.values,
    bar_width,
    edgecolor='gray',
    linewidth=1.2,
    color=colors,
    zorder=3
)

# 4. Zero baseline
ax.set_ylim(bottom=0)

# 5. Titles & labels
ax.set_title('Overall Hit Rates', fontsize=16, pad=15)
ax.set_ylabel('Hit Rate', fontsize=14, labelpad=10)
ax.set_xticks(index)
ax.set_xticklabels(series.index, rotation=0, fontsize=12)
ax.tick_params(axis='y', labelsize=12)

# 6. Light dashed gridlines
ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)

# 7. Annotate bars with percentages
for bar in bars:
    h = bar.get_height()
    ax.annotate(
        f'{h:.1%}',
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=11
    )

plt.tight_layout()
plt.show()

# 8. Print the numeric values
print('Overall rates:', {k: f"{v:.2%}" for k, v in rates.items()})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Helper functions
def compute_stats(vals):
    """Return mean, median, and mode of a list of values."""
    s = pd.Series(vals)
    mean = s.mean()
    median = s.median()
    mode = s.mode().iloc[0] if not s.mode().empty else None
    return mean, median, mode

def styled_hist(ax, data, bins, stats, title):
    """
    Draw a histogram on ax with pastel fill, dashed gridlines,
    and vertical lines for mean/median/mode.
    Parameters must be passed positionally:
      ax, data, bins, (mean, median, mode), title
    """
    mean, median, mode = stats
    # Soft pastel fill
    pastel = plt.get_cmap('Pastel1')
    color = pastel(0.5)
    ax.hist(data, bins=bins, align='left',
            edgecolor='gray', linewidth=1.2,
            color=color, zorder=3)
    ax.set_ylim(bottom=0)  # zero baseline
    ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
    # Add mean/median/mode lines
    for val, ls, lbl, col in [
        (mean, '--', f"Mean {mean:.1f}", 'red'),
        (median, ':', f"Median {median:.1f}", 'blue'),
        (mode, '-.', f"Mode {mode:.0f}", 'green')
    ]:
        ax.axvline(val, color=col, linestyle=ls, label=lbl)
    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel('Rank', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.legend(fontsize=10)


# 3. Prepare rank data
# 5.1 Combined Engines
all_ranks = []
nonhits = 0
for recs in data.values():
    for rec in recs.values():
        prompts = set(rec['urls_from_prompt'])
        found = {e['url'] for e in rec['bing_urls'] + rec['google_urls']}
        # collect ranks for all hits
        all_ranks.extend(
            e['rank']
            for e in rec['bing_urls'] + rec['google_urls']
            if e['url'] in prompts
        )
        nonhits += len(prompts - found)
# represent misses with rank=0
all_ranks += [0] * nonhits
stats_all = compute_stats(all_ranks)

# Plot 5.1
fig, ax = plt.subplots(figsize=(8, 5))
styled_hist(
    ax,
    all_ranks,
    range(0, max(all_ranks) + 2),
    stats_all,
    '5.1 Combined Engines Rank Distribution'
)
plt.tight_layout()
plt.show()

# 5.2 Per-Engine Distributions
for engine, title in [
    ('bing', '5.2 Bing Rank Distribution'),
    ('google', '5.2 Google Rank Distribution')
]:
    ranks = [
        e['rank']
        for recs in data.values()
        for rec in recs.values()
        for e in rec[f'{engine}_urls']
        if e['url'] in set(rec['urls_from_prompt'])
    ]
    stats = compute_stats(ranks)
    fig, ax = plt.subplots(figsize=(8, 5))
    styled_hist(
        ax,
        ranks,
        range(1, max(ranks) + 2),
        stats,
        title
    )
    plt.tight_layout()
    plt.show()

# 5.3 Per-Category Combined (2×3 grid)
categories = list(data.keys())
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, cat in zip(axes.flatten(), categories):
    cat_ranks = []
    miss_cat = 0
    for rec in data[cat].values():
        prompts = set(rec['urls_from_prompt'])
        found = set()
        for e in rec['bing_urls'] + rec['google_urls']:
            if e['url'] in prompts:
                cat_ranks.append(e['rank'])
                found.add(e['url'])
        miss_cat += len(prompts - found)
    combined = cat_ranks + [0] * miss_cat
    stats_cat = compute_stats(combined)
    styled_hist(
        ax,
        combined,
        range(0, max(combined or [0]) + 2),
        stats_cat,
        f'{cat} Combined'
    )
# turn off any extra subplots
for ax in axes.flatten()[len(categories):]:
    ax.axis('off')
fig.suptitle('5.3 Rank Distribution by Category (Combined)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# 5.4 Per-Category, Per-Engine
for cat in categories:
    for engine in ('bing', 'google'):
        ranks = [
            e['rank']
            for rec in data[cat].values()
            for e in rec[f'{engine}_urls']
            if e['url'] in set(rec['urls_from_prompt'])
        ]
        if not ranks:
            continue
        stats_engine = compute_stats(ranks)
        fig, ax = plt.subplots(figsize=(8, 5))
        styled_hist(
            ax,
            ranks,
            range(1, max(ranks) + 2),
            stats_engine,
            f'5.4 {engine.title()} Rank Distribution for {cat}'
        )
        plt.tight_layout()
        plt.show()


In [ ]:
# 6) Hit rate for cited URLs per category
cited_records = []
overall_cited = {'hits': 0, 'total': 0}
for cat, har_dict in data.items():
    for rec in har_dict.values():
        cited = {strip_utm(u) for u in rec.get('urls_cited', [])}
        found_urls = {strip_utm(e['url']) for e in rec['bing_urls'] + rec['google_urls']}
        hits_c = len(cited & found_urls)
        total_c = len(cited)
        overall_cited['hits'] += hits_c
        overall_cited['total'] += total_c
        if total_c > 0:
            cited_records.append({'category': cat, 'rate_cited': hits_c / total_c})
# Average cited hit rate per category + overall
if cited_records:
    df_cr = pd.DataFrame(cited_records)
    avg_cr = df_cr.groupby('category')['rate_cited'].mean().sort_index()
    # include overall
    overall_rate = overall_cited['hits'] / overall_cited['total'] if overall_cited['total'] else 0
    avg_cr_with_overall = avg_cr.append(pd.Series({'Overall': overall_rate}))
    avg_cr_with_overall.plot(kind='bar', edgecolor='black')
    plt.ylabel('Avg Cited Hit Rate')
    plt.title('Cited URL Hit Rate per Category + Overall')
    plt.tight_layout()
    plt.show()
print('Overall cited hit rate:', f"{overall_cited['hits']/overall_cited['total']:.2%}" if overall_cited['total'] else None)
